# 💰 SmartStock – Profit & Financial Health Analytics
### Revenue, COGS, Gross Profit, Expenses, and Net Profit Waterfall
**Objective**: Deliver dynamic P&L financial reporting, gross margin tracking, operating expense breakdown, and bottom-line Net Profit analytics.


## 1. Environment Setup & Financial Ingestion


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = os.path.join('..', 'datasets')
df_sales = pd.read_csv(os.path.join(DATA_DIR, 'sales.csv'))
df_items = pd.read_csv(os.path.join(DATA_DIR, 'sale_items.csv'))
df_expenses = pd.read_csv(os.path.join(DATA_DIR, 'expenses.csv'))

df_sales['sale_date'] = pd.to_datetime(df_sales['sale_date'])
df_sales['month'] = df_sales['sale_date'].dt.to_period('M')

df_expenses['expense_date'] = pd.to_datetime(df_expenses['expense_date'])
df_expenses['month'] = df_expenses['expense_date'].dt.to_period('M')

print("Loaded sales and expense records successfully.")


## 2. Executive Profit & Loss (P&L) Statement
Dynamic formulation based on enterprise formulas:
* **Revenue**: Total Gross Sales (`total_amount.sum()`)
* **COGS**: Cost of Goods Sold (`sum(quantity * unit_cost)`)
* **Gross Profit**: `Revenue - COGS`
* **Operating Expenses**: `sum(expenses.amount)`
* **Net Profit**: `Gross Profit - Expenses`
* **Net Profit Margin**: `(Net Profit / Revenue) * 100`


In [ ]:
total_revenue = df_sales['total_amount'].sum()
total_cogs = df_items['cogs'].sum()
gross_profit = total_revenue - total_cogs
total_expenses = df_expenses['amount'].sum()
net_profit = gross_profit - total_expenses

gross_margin_pct = (gross_profit / total_revenue * 100) if total_revenue > 0 else 0.0
net_margin_pct = (net_profit / total_revenue * 100) if total_revenue > 0 else 0.0

pnl_summary = pd.DataFrame([
    {"Financial Line Item": "1. Gross Revenue", "Amount ($)": total_revenue, "% of Revenue": "100.0%"},
    {"Financial Line Item": "2. Cost of Goods Sold (COGS)", "Amount ($)": -total_cogs, "% of Revenue": f"{(total_cogs/total_revenue)*100:.1f}%"},
    {"Financial Line Item": "3. GROSS PROFIT", "Amount ($)": gross_profit, "% of Revenue": f"{gross_margin_pct:.1f}%"},
    {"Financial Line Item": "4. Operating Expenses (OPEX)", "Amount ($)": -total_expenses, "% of Revenue": f"{(total_expenses/total_revenue)*100:.1f}%"},
    {"Financial Line Item": "5. NET PROFIT", "Amount ($)": net_profit, "% of Revenue": f"{net_margin_pct:.1f}%"}
])

display(pnl_summary)

print("=" * 60)
print(f"💵 Total Revenue:        ${total_revenue:>12,.2f}")
print(f"📦 Total COGS:           ${total_cogs:>12,.2f}")
print(f"📈 Gross Profit:         ${gross_profit:>12,.2f}  (Margin: {gross_margin_pct:.1f}%)")
print(f"📉 Operating Expenses:   ${total_expenses:>12,.2f}")
print(f"🏆 Net Profit:           ${net_profit:>12,.2f}  (Margin: {net_margin_pct:.1f}%)")
print("=" * 60)


## 3. Profit Waterfall Breakdown
Visualize the financial flow from top-line revenue through COGS and operating expenses to final net income.


In [ ]:
waterfall_labels = ['Revenue', 'COGS', 'Gross Profit', 'Expenses', 'Net Profit']
waterfall_values = [total_revenue, -total_cogs, gross_profit, -total_expenses, net_profit]
bar_colors = ['#0ea5e9', '#ef4444', '#10b981', '#f97316', '#8b5cf6']

plt.figure(figsize=(10, 6))
bars = plt.bar(waterfall_labels, [abs(v) for v in waterfall_values], color=bar_colors, width=0.55, alpha=0.9)
plt.title('SmartStock Financial Waterfall (Revenue to Net Profit)', fontsize=14, fontweight='bold', pad=12)
plt.ylabel('Amount ($)', fontweight='semibold')
plt.gca().yaxis.set_major_formatter('${x:,.0f}')

for bar, val in zip(bars, waterfall_values):
    prefix = "+" if val > 0 else "-"
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 2000, f"{prefix}${abs(val):,.0f}", ha='center', fontweight='bold', fontsize=9.5)

plt.tight_layout()
plt.show()


## 4. Operating Expenses Breakdown by Category
Analyze where operational capital is disbursed.


In [ ]:
exp_cat = df_expenses.groupby('category')['amount'].sum().sort_values(ascending=False).reset_index()
exp_cat['share_pct'] = (exp_cat['amount'] / exp_cat['amount'].sum()) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Donut chart
ax1.pie(exp_cat['amount'], labels=exp_cat['category'], autopct='%1.1f%%',
        colors=['#6366f1', '#ec4899', '#f59e0b', '#10b981', '#06b6d4', '#8b5cf6', '#64748b', '#e11d48'],
        startangle=140, pctdistance=0.8, wedgeprops=dict(width=0.4, edgecolor='w'))
ax1.set_title('Expense Distribution by Category', fontweight='bold')

# Horizontal bar
ax2.barh(exp_cat['category'][::-1], exp_cat['amount'][::-1], color='#f43f5e', alpha=0.85)
ax2.set_title('Expense Amount by Category ($)', fontweight='bold')
ax2.xaxis.set_major_formatter('${x:,.0f}')

plt.tight_layout()
plt.show()

display(exp_cat)


## 5. Monthly Net Profit Progression & Margin Sustainability
Track month-by-month Gross vs. Net margins over time.


In [ ]:
m_sales = df_sales.groupby('month').agg(revenue=('total_amount', 'sum')).reset_index()
m_items = df_items.groupby(df_items['sale_date'].dt.to_period('M')).agg(cogs=('cogs', 'sum')).reset_index()
m_items.rename(columns={'sale_date': 'month'}, inplace=True)
m_exp = df_expenses.groupby('month').agg(expenses=('amount', 'sum')).reset_index()

m_pnl = pd.merge(m_sales, m_items, on='month', how='left')
m_pnl = pd.merge(m_pnl, m_exp, on='month', how='left').fillna(0)

m_pnl['gross_profit'] = m_pnl['revenue'] - m_pnl['cogs']
m_pnl['net_profit'] = m_pnl['gross_profit'] - m_pnl['expenses']
m_pnl['net_margin_pct'] = (m_pnl['net_profit'] / m_pnl['revenue']) * 100

display(m_pnl)

plt.figure(figsize=(12, 6))
x_axis = [str(m) for m in m_pnl['month']]
plt.plot(x_axis, m_pnl['revenue'], marker='o', label='Gross Revenue', color='#0ea5e9', lw=2.5)
plt.plot(x_axis, m_pnl['gross_profit'], marker='s', label='Gross Profit', color='#10b981', lw=2.5)
plt.plot(x_axis, m_pnl['expenses'], marker='^', label='Operating Expenses', color='#f97316', lw=2)
plt.plot(x_axis, m_pnl['net_profit'], marker='D', label='Net Profit', color='#8b5cf6', lw=3)

plt.title('Monthly Financial Performance Trend (Revenue, OPEX, Net Profit)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Month', fontweight='semibold')
plt.ylabel('Amount ($)', fontweight='semibold')
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()
